# Fraud Detection – Exploratory Data Analysis (EDA)

Initial exploration of the IEEE-CIS dataset to understand structure, class balance, missing values, and feature patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython.display import display

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
os.makedirs('../results/figures', exist_ok=True)

## 1. Load & Merge Data

In [ ]:
train_trans = pd.read_csv('../data/raw/train_transaction.csv')
train_id    = pd.read_csv('../data/raw/train_identity.csv')

print('Transaction shape:', train_trans.shape)
print('Identity shape:   ', train_id.shape)

df = train_trans.merge(train_id, on='TransactionID', how='left')
print('Merged shape:     ', df.shape)


## 2. Basic Exploration

In [ ]:
display(df.head())
print('='*60)
print(df.dtypes.value_counts())
print('='*60)
display(df.describe(include='all').T.head(20))


## 3. Target Variable – Class Imbalance

In [ ]:
vc = df['isFraud'].value_counts()
print('Value counts:\n', vc)
print('\nFraud rate: {:.2%}'.format(vc[1] / len(df)))

ax = sns.countplot(x='isFraud', data=df, palette='viridis')
ax.set(title='Fraud vs Non-Fraud Distribution', xlabel='isFraud', ylabel='Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x()+p.get_width()/2, p.get_height()+500),
                ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../results/figures/fraud_distribution.png', dpi=150)
plt.show()


## 4. Missing Values Analysis

In [ ]:
missing_pct = df.isnull().mean() * 100
missing_df  = missing_pct[missing_pct > 0].sort_values(ascending=False)
print(f'Features with missing values: {len(missing_df)}/{len(df.columns)}')
display(missing_df.head(20).to_frame('missing_%'))

missing_df.head(20).plot(kind='bar', color='steelblue')
plt.title('Top 20 Features with Missing Values (%)')
plt.ylabel('Missing %')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../results/figures/missing_values.png', dpi=150)
plt.show()

cols_drop_90 = missing_pct[missing_pct > 90].index
print(f'\nColumns >90% missing ({len(cols_drop_90)} total): {cols_drop_90.tolist()}')


## 5. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['TransactionAmt'], bins=100, ax=axes[0])
axes[0].set(title='TransactionAmt (Raw)', xlabel='Amount')

df['_log_amt'] = np.log1p(df['TransactionAmt'])
sns.histplot(df['_log_amt'], bins=100, ax=axes[1], color='darkorange')
axes[1].set(title='TransactionAmt (Log-Transformed)', xlabel='log1p(Amount)')
plt.tight_layout()
plt.savefig('../results/figures/transaction_amt_raw.png', dpi=150)
plt.show()

sns.histplot(df['_log_amt'], bins=100, color='darkorange')
plt.title('Transaction Amount (Log Transformed)')
plt.savefig('../results/figures/transaction_amt_log.png', dpi=150)
plt.show()
df.drop(columns='_log_amt', inplace=True)


## 6. Transaction Amount vs Fraud

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='isFraud', y='TransactionAmt', data=df, ax=axes[0], palette='Set2')
axes[0].set(title='Raw Amount by Fraud')
sns.boxplot(x='isFraud', y=np.log1p(df['TransactionAmt']), data=df, ax=axes[1], palette='Set2')
axes[1].set(title='Log Amount by Fraud', ylabel='log1p(TransactionAmt)')
plt.tight_layout()
plt.savefig('../results/figures/transaction_amt_fraud.png', dpi=150)
plt.show()


## 7. Categorical Feature Analysis

In [ ]:
cat_features = ['ProductCD', 'card4', 'DeviceType']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, cat_features):
    if col in df.columns:
        top = df[col].value_counts().index[:6]
        sns.countplot(data=df[df[col].isin(top)], x=col, hue='isFraud', ax=ax, palette='Set1')
        ax.set_title(f'{col} vs Fraud')
        ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../results/figures/categorical_fraud.png', dpi=150)
plt.show()


## 8. Fraud Rate Over Time

In [ ]:
df['TransactionDT_hours'] = df['TransactionDT'] / 3600
df.groupby('TransactionDT_hours')['isFraud'].mean().rolling(50).mean().plot(color='crimson')
plt.title('Fraud Rate Over Time (Smoothed)')
plt.xlabel('Time (hours)')
plt.ylabel('Fraud Rate')
plt.tight_layout()
plt.savefig('../results/figures/fraud_rate_time.png', dpi=150)
plt.show()


## 9. Correlation with Target

In [ ]:
sample_df   = df.sample(20000, random_state=42)
corr_target = sample_df.corr(numeric_only=True)['isFraud'].drop('isFraud').sort_values(key=abs, ascending=False)
display(corr_target.head(20).to_frame('corr_with_fraud'))

corr_target.head(20).plot(kind='bar', color='teal')
plt.title('Top 20 Features Correlated with Fraud')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../results/figures/corr_with_fraud.png', dpi=150)
plt.show()


## 10. Decimal Extraction Feature

In [ ]:
df['TransactionAmt_decimal'] = (df['TransactionAmt'] - df['TransactionAmt'].astype(int)) * 1000
display(df[['TransactionAmt', 'TransactionAmt_decimal']].head(10))


## 11. Feature Type Summary

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
print(f'Categorical features : {len(cat_cols)}')
print(f'Numerical features   : {len(num_cols)}')
print('\nKey Insights:')
print('  - 3.5% fraud rate → severe class imbalance → use stratified split + class weights')
print('  - 12 features have >90% missing → drop them')
print('  - TransactionAmt is right-skewed → log-transform')
print('  - Most features have weak linear correlation → complex interactions matter')
